# Neural Network (MLP)

Multi-layer Perceptron — a feedforward neural network capable of learning complex non-linear patterns. This template uses sklearn's MLP implementation (fits the standard Pipeline/GridSearchCV workflow) with an optional PyTorch section for custom architectures.

**When to Use:**
- Complex non-linear relationships that simpler models cannot capture
- Large datasets where overfitting risk is manageable
- When other models (tree-based, linear) have plateaued
- Tabular data with important feature interactions

**Key Assumptions / Considerations:**
- Requires feature scaling (critical for convergence)
- Architecture (hidden layers, neurons) requires tuning
- More data-hungry than tree-based models — may overfit on small datasets
- Training can be slow without GPU (sklearn MLP is CPU-only)
- Early stopping recommended to prevent overfitting
- Less interpretable than linear models or decision trees

**References:**
- [sklearn MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)
- [sklearn MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)
- [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data 

np.random.seed(100)
n_samples = 8000

age = np.random.randint(18, 70, n_samples)
income = np.random.normal(50000, 15000, n_samples)
num_transactions = np.random.poisson(10, n_samples)

gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

target_continuous = 0.05 * age + 0.0005 * income + 0.3 * num_transactions + np.random.normal(0, 2, n_samples)
target_binary = (target_continuous > np.median(target_continuous)).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
    'target': target_continuous  # switch to target_binary for classification 
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Distribution

plt.figure(figsize=(8, 5))
sns.histplot(y, kde=True, bins=30)
plt.title("Target Variable Distribution")
plt.show()

print("Skewness:", round(y.skew(),4))
print("Kurtosis:", round(y.kurt(), 4))

In [ ]:
# Train/Test Split

try:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    # if y is continuous
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

In [ ]:
# Preprocessing

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Parameter Grids

models = {
    "MLP_Regressor": (MLPRegressor(random_state=42, max_iter=500, early_stopping=True, validation_fraction=0.1), {
        "clf__hidden_layer_sizes": [(64,), (128, 64), (128, 64, 32)],
        "clf__activation": ["relu", "tanh"],
        "clf__alpha": [1e-4, 1e-3, 1e-2],
        "clf__learning_rate": ["constant", "adaptive"],
        "clf__learning_rate_init": [0.001, 0.01],
    }),
    "MLP_Classifier": (MLPClassifier(random_state=42, max_iter=500, early_stopping=True, validation_fraction=0.1), {
        "clf__hidden_layer_sizes": [(64,), (128, 64), (128, 64, 32)],
        "clf__activation": ["relu", "tanh"],
        "clf__alpha": [1e-4, 1e-3, 1e-2],
        "clf__learning_rate": ["constant", "adaptive"],
        "clf__learning_rate_init": [0.001, 0.01],
    }),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1) if len(y.unique()) < 10 else KFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Training

def train_models(X_train, y_train, X_test, y_test):

    results = []
    fitted_pipelines = []

    for name, (model, params) in models.items():
        print(f"\n🔹 Training {name}...")

        is_classifier = isinstance(model, MLPClassifier)

        pipe = Pipeline([("prep", preprocessor), ("clf", model)])

        scoring = "roc_auc" if is_classifier else "r2"

        grid = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring=scoring, return_train_score=False)

        try:
            y_train_use = y_train if not is_classifier else (y_train > np.median(y_train)).astype(int)
            y_test_use = y_test if not is_classifier else (y_test > np.median(y_test)).astype(int)

            grid.fit(X_train, y_train_use)
            best_model = grid.best_estimator_
            fitted_pipelines.append((name, best_model))
            y_pred = best_model.predict(X_test)

            if is_classifier:
                y_proba = best_model.predict_proba(X_test)[:, 1]
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "Accuracy": accuracy_score(y_test_use, y_pred),
                    "Recall": recall_score(y_test_use, y_pred),
                    "Precision": precision_score(y_test_use, y_pred),
                    "F1 Score": f1_score(y_test_use, y_pred),
                    "ROC-AUC": roc_auc_score(y_test_use, y_proba),
                }
            else:
                n = len(y_test_use)
                p = X_train.shape[1]
                rss = np.sum((y_test_use - y_pred) ** 2)
                mse = mean_squared_error(y_test_use, y_pred)
                aic = n * np.log(rss / n) + 2 * p
                bic = n * np.log(rss / n) + p * np.log(n)
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "RMSE": np.sqrt(mse),
                    "MAE": mean_absolute_error(y_test_use, y_pred),
                    "R² Score": r2_score(y_test_use, y_pred),
                    "AIC": aic,
                    "BIC": bic,
                }

            results.append(metrics)
        except Exception as e:
            print(f"⚠️ Skipping {name} due to error: {e}")
            continue

    return results, fitted_pipelines

In [ ]:
results, pipelines = train_models(X_train_full, y_train_full, X_test, y_test)

In [ ]:
# Results Summary

results_df = pd.DataFrame(results)
print("\n✅ Model Evaluation Summary:")
results_df

In [ ]:
# Best Regression Model by R²
regression_results = [r for r in results if 'R² Score' in r]
best_model_name = max(regression_results, key=lambda x: x['R² Score'])['Model']
best_model_index = [i for i, (n, p) in enumerate(pipelines) if n == best_model_name][0]
best_model_pipeline = pipelines[best_model_index][1]

print(f"\n🏆 Best Regression Model: {best_model_name}")

In [ ]:
# Learning Curves

for name, pipe in pipelines:
    mlp = pipe.named_steps["clf"]
    
    if hasattr(mlp, "loss_curve_"):
        plt.figure(figsize=(10, 5))
        plt.plot(mlp.loss_curve_, label="Training Loss")
        
        if hasattr(mlp, "validation_scores_") and mlp.validation_scores_ is not None:
            plt.plot(mlp.validation_scores_, label="Validation Score")
        
        plt.xlabel("Iteration")
        plt.ylabel("Loss / Score")
        plt.title(f"{name} - Learning Curve")
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# First Layer Weight Visualization

for name, pipe in pipelines:
    mlp = pipe.named_steps["clf"]
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    # First layer weights
    weights = mlp.coefs_[0]  # shape: (n_features, n_hidden_1)
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(weights, xticklabels=[f"H{i}" for i in range(weights.shape[1])],
                yticklabels=feature_names, cmap="coolwarm", center=0)
    plt.title(f"{name} - First Layer Weights")
    plt.xlabel("Hidden Neurons")
    plt.ylabel("Input Features")
    plt.tight_layout()
    plt.show()
    
    break  # show first model only

In [ ]:
# Regression Diagnostics

def regression_diagnostics(model, X, y, name="Model"):
    y_pred = model.predict(X)
    residuals = y - y_pred
    fitted = y_pred

    plt.figure(figsize=(12, 10))
    
    plt.subplot(2,2,1)
    sns.scatterplot(x=fitted, y=residuals)
    plt.axhline(0, color='r', linestyle='--')
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"{name} - Residuals vs Fitted")
    
    plt.subplot(2,2,2)
    sns.histplot(residuals, kde=True, bins=30)
    plt.title(f"{name} - Residual Distribution")
    
    plt.subplot(2,2,3)
    sns.scatterplot(x=fitted, y=np.sqrt(np.abs(residuals)))
    plt.xlabel("Fitted values")
    plt.ylabel("Sqrt(|Residuals|)")
    plt.title(f"{name} - Scale-Location")
    
    plt.subplot(2,2,4)
    sns.scatterplot(x=y, y=fitted)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{name} - Predicted vs Actual")
    
    plt.tight_layout()
    plt.show()

for name, pipe in pipelines:
    if "Regressor" in name:
        regression_diagnostics(pipe, X_test, y_test, f"{name} (Test)")

In [ ]:
# Classification Diagnostics

for name, pipe in pipelines:
    if "Classifier" in name:
        y_test_binary = (y_test > np.median(y_test)).astype(int)
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        cm = confusion_matrix(y_test_binary, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")
        axes[0].set_title(f"{name} - Confusion Matrix")
        
        fpr, tpr, _ = roc_curve(y_test_binary, y_proba)
        auc = roc_auc_score(y_test_binary, y_proba)
        axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        axes[1].plot([0, 1], [0, 1], 'r--')
        axes[1].set_xlabel("False Positive Rate")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].set_title(f"{name} - ROC Curve")
        axes[1].legend()
        
        axes[2].hist(y_proba[y_test_binary == 0], bins=30, alpha=0.5, label="Class 0")
        axes[2].hist(y_proba[y_test_binary == 1], bins=30, alpha=0.5, label="Class 1")
        axes[2].set_xlabel("Predicted Probability")
        axes[2].set_title(f"{name} - Probability Distribution")
        axes[2].legend()
        
        plt.tight_layout()
        plt.show()
        
        print(classification_report(y_test_binary, y_pred))

In [ ]:
# Profile Plots

def plot_profiles(best_pipeline, X, y_true):
    y_pred = best_pipeline.predict(X)
    
    n_cols = 3
    n_features = X.shape[1]
    n_rows = math.ceil(n_features / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()  

    for i, col in enumerate(X.columns):
        df = pd.DataFrame({
            col: X[col],
            "y_true": y_true,
            "y_pred": y_pred
        })

        grouped = df.groupby(col).agg(
            count=("y_true", "size"),
            avg_true=("y_true", "mean"),
            avg_pred=("y_pred", "mean")
        ).reset_index().sort_values(col)

        ax1 = axes[i]
        ax1.bar(grouped[col], grouped["count"], alpha=0.3)
        ax1.set_xlabel(col)
        ax1.set_ylabel("Population")

        ax2 = ax1.twinx()
        ax2.plot(grouped[col], grouped["avg_true"], marker="o", label="Actual Target")
        ax2.plot(grouped[col], grouped["avg_pred"], marker="o", linestyle="--", label="Predicted Target")
        ax2.set_ylabel("Target Value")

        ax1.set_title(col)
        ax2.legend(loc="upper right")


    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_profiles(best_model_pipeline, X_train_full, y_train_full)

## PyTorch Alternative

The section below shows an equivalent neural network built with PyTorch for users who need custom architectures, GPU training, or more control over the training loop. Requires `pip install torch`.

In [ ]:
# pip install torch

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch not installed. Run: pip install torch")

if TORCH_AVAILABLE:
    # Prepare data
    X_train_tensor = torch.FloatTensor(preprocessor.fit_transform(X_train_full))
    X_test_tensor = torch.FloatTensor(preprocessor.transform(X_test))
    y_train_tensor = torch.FloatTensor(y_train_full.values).unsqueeze(1)
    y_test_tensor = torch.FloatTensor(y_test.values).unsqueeze(1)
    
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    
    # Define model
    class SimpleNet(nn.Module):
        def __init__(self, input_dim, hidden_dims=[128, 64, 32]):
            super().__init__()
            layers = []
            prev_dim = input_dim
            for h in hidden_dims:
                layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(0.2)])
                prev_dim = h
            layers.append(nn.Linear(prev_dim, 1))
            self.network = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.network(x)
    
    input_dim = X_train_tensor.shape[1]
    model = SimpleNet(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # Training loop with early stopping
    n_epochs = 100
    patience = 10
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        train_losses.append(epoch_loss / len(train_loader))
        
        model.eval()
        with torch.no_grad():
            val_output = model(X_test_tensor)
            val_loss = criterion(val_output, y_test_tensor).item()
            val_losses.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"✅ Early stopping at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    
    # Plot learning curves
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("PyTorch Neural Network - Learning Curves")
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        y_pred_torch = model(X_test_tensor).numpy().flatten()
    
    print(f"\n🏆 PyTorch Model Results:")
    print(f"   RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_torch)):.4f}")
    print(f"   MAE:  {mean_absolute_error(y_test, y_pred_torch):.4f}")
    print(f"   R²:   {r2_score(y_test, y_pred_torch):.4f}")

In [ ]:
# future work:
#   GPU training with PyTorch (model.to('cuda'))
#   batch normalization for faster convergence
#   learning rate schedulers (ReduceLROnPlateau, CosineAnnealing)
#   architecture search (wider vs deeper networks)
#   dropout tuning for regularization
#   attention mechanisms for tabular data (TabNet)
#   hyperparameter optimization with Optuna